# Queueing theory - E2/E2/1 example

An example of a queue in which both the interarrival times and the service times are Erlang 2 distributed. The simulated averages are compared with the M/M/1 queue with the same rates and with Kingman's approximation for G/G/1 queues.

The service station is composed of mix-ins, in the same way as objects in OpenTNSim. Because customers are generated by the arrival and service processes (rates, not a list of known customers), only the service station is created as an object.

This notebook provides an example how to set up a queueing simulation. Input that needs to be specified are:
* 'Lambda': the number of arrivals per hour
* 'Mu': the number of departures (services completed) per server per hour
* 'nr_resources': the number of servers of the service station (c in Kendall's notation)

To further characterise the queueing behaviour in the system the user has to specify the probability distributions that characterise the interarrival and service times. This notebook describes an E2/E2/1 queue (according to the Kendall notation for queueing systems) where both the interarrival times and the service times are assumed to be Erlang 2 distributed.

Currently implemented distributions are:
* 'M': Negative exponential distribution,
* 'Ek': Erlang k distributions (where k needs to be an integer number, e.g. E2, E3 etc).
* 'D': Deterministic lists with known input

NB: the calculation becomes more accurate when a larger number of arrivals (nr_arr) is simulated. Obviously there is a tradeoff between accuracy and calculation time.

In [ ]:
# OpenQTSim mixins and utils
import openqtsim
from openqtsim.core import Identifiable, HasResource
from openqtsim.core import create_object, inspect_object
import pandas as pd
from openqtsim.visualizations import plot_gantt

print("This notebook is executed with OpenQTSim version {}".format(openqtsim.__version__))

#### 0. Define input

In [ ]:
example_title = "E2/E2/1"
Lambda = 8  # nr of arrivals per hour
Mu = 9  # nr of exits (services completed) per server per hour
nr_arr = 100_000
arrival_seed = 42  # random stream of the interarrival times
service_seed = 43  # independent random stream of the service times

#### 1. Create service station
Next we create a Server-object (our service station) out of the available mix-ins:

In [ ]:
# make your preferred Server class out of available mix-ins.
Server = create_object(
    "Server",
    (
        Identifiable,  # allows to give the object a name and a random ID
        HasResource,   # number of servers (c in Kendall's notation)
    ),
)

We can inspect the object to determine which inputs are required and optional:

In [ ]:
inspect_object(Server, show_parameter_table=True)

We define the required input and create the server object:

In [ ]:
data_server = {
    "name": "Server",     # required by Identifiable
    "nr_resources": 1,    # optional for HasResource: one server
}

server = Server(**data_server)

#### 2. Define arrival and service processes
Customers are generated from Erlang 2 distributed interarrival and service times:

In [ ]:
A = openqtsim.ArrivalProcess("E2", arr_rate=Lambda)
S = openqtsim.ServiceProcess("E2", srv_rate=Mu)

#### 3. Run simulation
The server object sets the number of servers of the queue.

In [ ]:
%%time
q = openqtsim.Queue(A, S, station=server)
print('Start simulation of "{}" queue'.format(q.kendall_notation))

sim = openqtsim.Simulation(q, arrival_seed=arrival_seed, service_seed=service_seed)
sim.run(nr_arr)

In [ ]:
summary = sim.get_stats()
print('Input IAT: {:.4f}'.format(1/Lambda))
print('Input ST: {:.4f}'.format(1/Mu))

#### 4. Compare with M/M/1 and Kingman's approximation

There is no simple closed-form solution for the E2/E2/1 queue. A well-known approximation for a single-server G/G/1 queue is Kingman's formula:

$$W_q \approx \frac{c_a^2 + c_s^2}{2} \cdot \frac{\rho}{1-\rho} \cdot \frac{1}{\mu}$$

where $c_a$ and $c_s$ are the coefficients of variation of the interarrival and service times. For an Erlang k distribution $c^2 = 1/k$, so for E2 $c^2 = 0.5$ (for M $c^2 = 1$, which gives the exact M/M/1 result). $L_q = \lambda W_q$, $W_s = W_q + 1/\mu$ and $L_s = \lambda W_s$ follow from Little's law.

In [ ]:
rho = Lambda / Mu
k_arrival, k_service = 2, 2  # Erlang shape parameters (E2)
ca2, cs2 = 1 / k_arrival, 1 / k_service

def gg1_measures(ca2, cs2):
    W_q = (ca2 + cs2) / 2 * rho / (1 - rho) / Mu
    W_s = W_q + 1 / Mu
    return {"IAT": 1 / Lambda, "ST": 1 / Mu, "Rho_server": rho,
            "L_s": Lambda * W_s, "L_q": Lambda * W_q, "W_s": W_s, "W_q": W_q}

kingman = gg1_measures(ca2, cs2)
mm1 = gg1_measures(1, 1)
simulated = {key: summary[key] for key in kingman}

comparison = pd.DataFrame({"M/M/1 (exact)": mm1, "E2/E2/1 Kingman (approx.)": kingman, "E2/E2/1 simulated": simulated})
comparison["simulated vs Kingman (%)"] = 100 * (comparison["E2/E2/1 simulated"] / comparison["E2/E2/1 Kingman (approx.)"] - 1)
comparison.round(4)

Less variability in the interarrival and service times roughly halves the waiting time compared to the M/M/1 queue with the same utilisation. Kingman's formula is a heavy-traffic approximation; for this example it slightly overestimates the simulated waiting time. At a high utilisation ($\rho \approx 0.89$) successive waiting times are strongly correlated, so changing the seeds or the number of arrivals gives an impression of the remaining statistical uncertainty.

In [ ]:
sim.plot_system_state()

In [ ]:
sim.plot_IAT_ST()

#### 5. Inspect output

In [ ]:
df, df_sys = sim.return_log()
df.head(10)

# Meaning of the columns:
# c_id: customer id
# IAT: interarrival time 
# ST: service time 
# AT: arrival time 
# TSB: time service begins
# TSE: time service ends 
# TCSS: time customer spends in the system
# TCWQ: time customer waits int he queue
# ITS: idle time server
# s_id: id of server assigned to customer

In [ ]:
df_sys.head(10)

# Meaning of the columns:
# t: time of logging (if 'lambda' and 'mu' are specified as values per hour, the unit of 't' is hours)
# c_s: nr of customers in the system
# c_q nr of customers in the queue

#### 6. Gantt chart: first 20 customers

Only the first 20 customers are shown. Orange spans arrival (`AT`) to service start (`TSB`); blue spans service start to departure (`TSE`). Hover to inspect the times in hours (and days) since simulation start.

In [ ]:
first = df.sort_values(["AT", "c_id"]).head(20)
fig = plot_gantt(first, title=f"{example_title}: first {len(first)} of {len(df):,} customers "
                             f"(arrival seed {arrival_seed}, service seed {service_seed})")
fig.show()
first[["c_id", "AT", "TSB", "TSE", "TCWQ", "ST", "TCSS", "s_id"]]